
## 1. Imports and Setup
We import necessary libraries

In [32]:
import pickle
from pathlib import Path
import numpy as np
import pandas as pd

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Scikit-Learn (For Random Forest & Preprocessing)
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

# Config
DATA_PROCESSED = Path("../data/processed")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {DEVICE}")

Using device: cpu


## 2. Data Loading & Feature Engineering
We load the clean data and define two helper functions:
1. `compute_fft_features`: Converts time-series into frequency domain (for MLPs).
2. `prepare_tof_images`: Reshapes TOF data into 5-channel images (for CNNs).

In [33]:
# Load Data
with open(DATA_PROCESSED / "train_clean.pkl", "rb") as f:
    train_df = pickle.load(f)
with open(DATA_PROCESSED / "test_clean.pkl", "rb") as f:
    test_df = pickle.load(f)
with open(DATA_PROCESSED / "feature_map.pkl", "rb") as f:
    feature_map = pickle.load(f)

# Define Feature Sets
IMU_COLS = feature_map["imu_cols"]
THM_COLS = feature_map["thm_cols"]
TOF_COLS = feature_map["tof_cols"]
ALL_SENSORS = IMU_COLS + THM_COLS + TOF_COLS
IMU_THM = IMU_COLS + THM_COLS
LABEL_COL = feature_map["label_col"]

# Label Map
unique_labels = sorted(train_df[LABEL_COL].unique())
label_to_idx = {label: idx for idx, label in enumerate(unique_labels)}
num_classes = len(unique_labels)

# --- Function 1: FFT ---
def compute_fft_features(df, feature_cols, target_length=64):
    X_list, y_list = [], []
    grouped = df.groupby("sequence_id")
    for _, group in grouped:
        signals = group[feature_cols].values
        # Real FFT
        fft_mag = np.abs(np.fft.rfft(signals, axis=0))
        # Pad/Truncate
        if fft_mag.shape[0] < target_length:
            fft_mag = np.pad(fft_mag, ((0, target_length - fft_mag.shape[0]), (0, 0)))
        else:
            fft_mag = fft_mag[:target_length, :]
        X_list.append(fft_mag.flatten())
        y_list.append(label_to_idx[group[LABEL_COL].iloc[0]])
    return np.array(X_list), np.array(y_list)

# --- Function 2: TOF Images ---
def prepare_tof_images(df, tof_cols, target_length=64):
    X_list, y_list = [], []
    grouped = df.groupby("sequence_id")
    for _, group in grouped:
        raw = group[tof_cols].values
        if raw.shape[0] < target_length:
            raw = np.pad(raw, ((0, target_length - raw.shape[0]), (0, 0)), constant_values=-1)
        else:
            raw = raw[:target_length, :]
        # Reshape to (Time, 5 sensors, 8, 8)
        reshaped = raw.reshape(target_length, 5, 8, 8)
        reshaped[reshaped == -1] = 0
        X_list.append(reshaped)
        y_list.append(label_to_idx[group[LABEL_COL].iloc[0]])
    return np.array(X_list), np.array(y_list)

print("Engineering functions ready.")

Engineering functions ready.


## 3. Dataset Generation
We create three datasets:
* **FFT All:** For Model 1 and Model 6.
* **FFT Sub (IMU+THM):** For Model 2 and Fusion Models.
* **TOF Images:** For Model 3 and Fusion Models.

In [34]:
print("Generating datasets... (This may take a moment)")

# 1. FFT All Sensors
X_train_fft_all, y_train = compute_fft_features(train_df, ALL_SENSORS)
X_test_fft_all, y_test = compute_fft_features(test_df, ALL_SENSORS)

# 2. FFT IMU+THM Only
X_train_fft_sub, _ = compute_fft_features(train_df, IMU_THM)
X_test_fft_sub, _ = compute_fft_features(test_df, IMU_THM)

# 3. TOF Images
X_train_tof, _ = prepare_tof_images(train_df, TOF_COLS)
X_test_tof, _ = prepare_tof_images(test_df, TOF_COLS)

# Standardization (Critical for MLP convergence)
scaler_all = StandardScaler()
X_train_fft_all = scaler_all.fit_transform(X_train_fft_all)
# Note: In production we would save scaler, here we fit on train.

scaler_sub = StandardScaler()
X_train_fft_sub = scaler_sub.fit_transform(X_train_fft_sub)

print(f"Dataset Shapes:")
print(f" - All Sensors FFT: {X_train_fft_all.shape}")
print(f" - IMU+THM FFT:     {X_train_fft_sub.shape}")
print(f" - TOF Images:      {X_train_tof.shape}")

Generating datasets... (This may take a moment)
Dataset Shapes:
 - All Sensors FFT: (6515, 21248)
 - IMU+THM FFT:     (6515, 768)
 - TOF Images:      (6515, 64, 5, 8, 8)


In [35]:
# Generic Dataset Wrapper
class SimpleDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

# Generic Training Loop
def train_model(model, train_loader, criterion, optimizer, name="Model", epochs=20):
    model = model.to(DEVICE)
    print(f"--- Training {name} ---")
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        
        if (epoch+1) % 5 == 0:
            print(f"Epoch {epoch+1}/{epochs} - Loss: {running_loss/len(train_loader):.4f}")
    return model

# Create Loaders
BATCH = 256
loader_fft_all = DataLoader(SimpleDataset(X_train_fft_all, y_train), batch_size=BATCH, shuffle=True)
loader_fft_sub = DataLoader(SimpleDataset(X_train_fft_sub, y_train), batch_size=BATCH, shuffle=True)
loader_tof = DataLoader(SimpleDataset(X_train_tof, y_train), batch_size=BATCH, shuffle=True)

## 4. Models 1 & 2: FFT-MLP
We define a standard 3-layer MLP architecture and train two versions of it:
* **Model 1:** Trained on ALL sensors (IMU+THM+TOF).
* **Model 2:** Trained only on IMU+THM (Used for Fusion).

In [36]:
# Architecture
class BFRB_MLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(BFRB_MLP, self).__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, num_classes)
        )
    def forward(self, x): return self.layers(x)

criterion = nn.CrossEntropyLoss()

# --- Train Model 1 (All Sensors) ---
model_1 = BFRB_MLP(X_train_fft_all.shape[1], num_classes)
opt_1 = optim.Adam(model_1.parameters(), lr=0.001)
model_1 = train_model(model_1, loader_fft_all, criterion, opt_1, "Model 1 (MLP All)")

# --- Train Model 2 (IMU+THM) ---
model_2 = BFRB_MLP(X_train_fft_sub.shape[1], num_classes)
opt_2 = optim.Adam(model_2.parameters(), lr=0.001)
model_2 = train_model(model_2, loader_fft_sub, criterion, opt_2, "Model 2 (MLP IMU+THM)")

--- Training Model 1 (MLP All) ---
Epoch 5/20 - Loss: nan
Epoch 10/20 - Loss: nan
Epoch 15/20 - Loss: nan
Epoch 20/20 - Loss: nan
--- Training Model 2 (MLP IMU+THM) ---
Epoch 5/20 - Loss: nan
Epoch 10/20 - Loss: nan
Epoch 15/20 - Loss: nan
Epoch 20/20 - Loss: nan


## 5. Model 3: CNN-BiLSTM (TOF Only)
This model uses a 2D CNN to process the TOF grids and a BiLSTM to model the temporal sequence.

In [37]:
class BFRB_CNN_BiLSTM(nn.Module):
    def __init__(self, num_classes):
        super(BFRB_CNN_BiLSTM, self).__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(5, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.Flatten()
        )
        self.projection = nn.Linear(64 * 8 * 8, 128)
        self.lstm = nn.LSTM(128, 128, batch_first=True, bidirectional=True)
        self.classifier = nn.Linear(256, num_classes)
        
    def forward(self, x):
        B, T, C, H, W = x.size()
        c_in = x.view(B * T, C, H, W)
        c_out = self.projection(self.cnn(c_in))
        r_in = c_out.view(B, T, -1)
        lstm_out, _ = self.lstm(r_in)
        return self.classifier(lstm_out[:, -1, :])

# Train Model 3
model_3 = BFRB_CNN_BiLSTM(num_classes)
opt_3 = optim.AdamW(model_3.parameters(), lr=0.001, weight_decay=0.0001)
model_3 = train_model(model_3, loader_tof, criterion, opt_3, "Model 3 (CNN-BiLSTM)")

--- Training Model 3 (CNN-BiLSTM) ---
Epoch 5/20 - Loss: nan
Epoch 10/20 - Loss: nan
Epoch 15/20 - Loss: nan
Epoch 20/20 - Loss: nan


## 6. Model 4: Late Fusion Ensemble
This model does not require separate training. 
It combines the predictions of **Model 2** (MLP IMU+THM) and **Model 3** (CNN-BiLSTM) during inference.
We will implement the fusion logic in `05_evaluation.ipynb`.

## 7. Model 5: Intermediate Fusion
This complex architecture trains the MLP branch and CNN branch simultaneously and fuses their internal embeddings (Intermediate Fusion) before the final classification layer.

In [38]:
# --- Dual Input Dataset for Fusion ---
class DualDataset(Dataset):
    def __init__(self, X_fft, X_tof, y):
        self.X_fft = torch.tensor(X_fft, dtype=torch.float32)
        self.X_tof = torch.tensor(X_tof, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.X_fft[idx], self.X_tof[idx], self.y[idx]

loader_dual = DataLoader(DualDataset(X_train_fft_sub, X_train_tof, y_train), batch_size=BATCH, shuffle=True)

# --- Architecture ---
class IntermediateFusion(nn.Module):
    def __init__(self, dim_fft, num_classes):
        super(IntermediateFusion, self).__init__()
        # Branch 1: MLP
        self.mlp_branch = nn.Sequential(
            nn.Linear(dim_fft, 128), nn.ReLU(),
            nn.Linear(128, 128)
        )
        # Branch 2: CNN
        self.cnn_branch = nn.Sequential(
            nn.Conv2d(5, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.Flatten()
        )
        self.cnn_proj = nn.Linear(64*8*8, 128)
        self.lstm = nn.LSTM(128, 128, batch_first=True, bidirectional=True)
        
        # Fusion Layer (128 from MLP + 256 from BiLSTM = 384)
        self.classifier = nn.Linear(384, num_classes)

    def forward(self, x_fft, x_tof):
        # MLP Forward
        emb_mlp = self.mlp_branch(x_fft)
        
        # CNN Forward
        B, T, C, H, W = x_tof.size()
        c_in = x_tof.view(B * T, C, H, W)
        c_out = self.cnn_proj(self.cnn_branch(c_in))
        r_in = c_out.view(B, T, -1)
        lstm_out, _ = self.lstm(r_in)
        emb_cnn = lstm_out[:, -1, :] 
        
        # Concatenate
        combined = torch.cat((emb_mlp, emb_cnn), dim=1)
        return self.classifier(combined)

# --- Train Model 5 ---
print("--- Training Model 5 (Intermediate Fusion) ---")
model_5 = IntermediateFusion(X_train_fft_sub.shape[1], num_classes).to(DEVICE)
opt_5 = optim.Adam(model_5.parameters(), lr=0.001)

for epoch in range(20):
    model_5.train()
    running_loss = 0.0
    for x1, x2, labels in loader_dual:
        x1, x2, labels = x1.to(DEVICE), x2.to(DEVICE), labels.to(DEVICE)
        opt_5.zero_grad()
        out = model_5(x1, x2)
        loss = criterion(out, labels)
        loss.backward()
        opt_5.step()
        running_loss += loss.item()
    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1}/20 - Loss: {running_loss/len(loader_dual):.4f}")

--- Training Model 5 (Intermediate Fusion) ---
Epoch 5/20 - Loss: nan
Epoch 10/20 - Loss: nan
Epoch 15/20 - Loss: nan
Epoch 20/20 - Loss: nan


## 8. Model 6: Random Forest Baseline
We train a standard Random Forest Classifier using Scikit-Learn on the FFT features of all sensors.

In [39]:
print("--- Training Model 6 (Random Forest) ---")
model_rf = RandomForestClassifier(n_estimators=100, random_state=42)
model_rf.fit(X_train_fft_all, y_train)
print("Random Forest Trained.")

--- Training Model 6 (Random Forest) ---
Random Forest Trained.


## 9. Save Models
We save all trained models to the processed data directory for use in the evaluation notebook.

In [40]:
# Save PyTorch Models
torch.save(model_1.state_dict(), DATA_PROCESSED / "model_1_mlp_all.pth")
torch.save(model_2.state_dict(), DATA_PROCESSED / "model_2_mlp_sub.pth")
torch.save(model_3.state_dict(), DATA_PROCESSED / "model_3_cnn.pth")
torch.save(model_5.state_dict(), DATA_PROCESSED / "model_5_fusion.pth")

# Save Random Forest
with open(DATA_PROCESSED / "model_6_rf.pkl", "wb") as f:
    pickle.dump(model_rf, f)

# Save Label Map
with open(DATA_PROCESSED / "label_map.pkl", "wb") as f:
    pickle.dump(label_to_idx, f)

print("All models successfully trained and saved!")

All models successfully trained and saved!
